In [1]:
import pymysql
from sqlalchemy import create_engine, text
import pandas as pd
import numpy as np
from google.cloud import bigquery
from dotenv import load_dotenv
import os
import re
 
load_dotenv()

True

In [2]:
# 구글 bigquery 연결을 만듭니다.
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = '../../gcp-credentials.json'

project_id = os.getenv('GCP_PROJECT_ID')
dataset = os.getenv('BQ_DATASET_HHP')

big_engine = create_engine(f'bigquery://{project_id}/{dataset}')

In [3]:
# raw데이터를 가져옵니다.
query = text("""
    select *
    from household_power.raw
""")

with big_engine.connect() as conn:
    result = conn.execute(query)
    df = pd.DataFrame(result.fetchall(), columns=result.keys())

E:\Data_Project\public-data-analysis\.venv\Lib\site-packages\google\cloud\bigquery\client.py:613: UserWarning: Cannot create BigQuery Storage client, the dependency google-cloud-bigquery-storage is not installed.
  warnings.warn(


CREATE TABLE household_power.fact(
	-- 차원 (Dimension) --
    year              INT64,
    month             INT64,
    quarter           INT64,             -- 파생: 분기
    season            STRING,            -- 파생: 계절
    season_tariff 	  STRING,			 -- 요금제 계절 구분
    
    sd_code           INT64,
    sd_name           STRING,
    sgg_name          STRING,   
    
    house_cnt         INT64,
    avg_usage_kwh     FLOAT64,           -- power_usage 리네이밍
    avg_bill_won      FLOAT64,             -- bill 리네이밍
    
    total_usage_kwh   FLOAT64,           -- house_cnt * avg_usage_kwh
    total_bill_won    FLOAT64,           -- house_cnt * avg_bill_won
    unit_price        FLOAT64,           -- avg_bill_won / avg_usage_kwh
    
    usage_yoy_pct     FLOAT64,           -- 전년 동월 대비 변화율
    usage_mom_pct     FLOAT64            -- 전월 대비 변화율
)

In [4]:
# 컬럼명 수정
df = df.rename(columns={'power_usage':'avg_usage_kwh', 'bill':'avg_bill_won'})

In [5]:
df.head()

,id,sd_code,year,month,sd_name,sgg_name,house_cnt,avg_usage_kwh,avg_bill_won
0,75,36,2013,5,세종특별자치시,,57469,209.84,24044.0
1,326,36,2013,6,세종특별자치시,,57556,212.42,23920.0
2,577,36,2013,7,세종특별자치시,,58728,212.24,24515.0
3,828,36,2013,8,세종특별자치시,,60133,245.70,34191.0
4,1079,36,2013,9,세종특별자치시,,60696,226.47,28739.0


In [6]:
# quarter 입력
df['quarter'] = (df['month'] - 1) // 3 + 1

# season 입력
season_map = {
    3: '봄', 4: '봄', 5: '봄',
    6: '여름', 7: '여름', 8: '여름',
    9: '가을', 10: '가을', 11: '겨울',
    12: '겨울', 1: '겨울', 2: '겨울'
}
df['season'] = df['month'].map(season_map)

# season 요금 입력
season_tariff_map = {
    1: '기타', 2: '기타', 3: '기타',
    4: '기타', 5: '기타', 6: '기타',
    7: '하계', 8: '하계',
    9: '기타', 10: '기타', 11: '기타', 12: '기타'
}
df['season_tariff'] = df['month'].map(season_tariff_map)

In [7]:
df['total_usage_kwh'] = df['house_cnt']*df['avg_usage_kwh']
df['total_bill_won'] = df['house_cnt']*df['avg_bill_won']
df['unit_price'] = df['avg_bill_won']/df['avg_usage_kwh']

In [8]:
df = df.drop(columns=['id'])

In [9]:
# 정렬 먼저
df = df.sort_values(['sd_code', 'sgg_name', 'year', 'month'])

# 전월 대비 (MoM)
df['usage_mom_pct'] = df.groupby(['sd_code', 'sgg_name'])['avg_usage_kwh'].pct_change() * 100

# 전년 동월 대비 (YoY) - 12개월 전과 비교
df['usage_yoy_pct'] = df.groupby(['sd_code', 'sgg_name'])['avg_usage_kwh'].pct_change(periods=12) * 100

In [10]:
df = df.replace([np.inf, -np.inf], np.nan)

In [11]:
df.head()

,sd_code,year,month,sd_name,sgg_name,house_cnt,avg_usage_kwh,avg_bill_won,quarter,season,season_tariff,total_usage_kwh,total_bill_won,unit_price,usage_mom_pct,usage_yoy_pct
277,11,2013,5,서울특별시,강남구,238588,240.33,33987.0,2,봄,기타,57339854.04,8.108890e+09,141.418050,NaN,NaN
278,11,2013,6,서울특별시,강남구,238692,257.30,39306.0,2,여름,기타,61415451.60,9.382028e+09,152.763311,7.061124,NaN
279,11,2013,7,서울특별시,강남구,240068,288.91,50997.0,3,여름,하계,69358045.88,1.224275e+10,176.515178,12.285270,NaN
280,11,2013,8,서울특별시,강남구,240182,348.78,75578.0,3,여름,하계,83770677.96,1.815248e+10,216.692471,20.722716,NaN
281,11,2013,9,서울특별시,강남구,240363,287.54,49276.0,3,가을,기타,69113977.02,1.184413e+10,171.370940,-17.558346,NaN


In [12]:
# BigQuery 적재
client = bigquery.Client(project=project_id)
table_id = f'{project_id}.{dataset}.fact'

job_config = bigquery.LoadJobConfig(
    write_disposition='WRITE_TRUNCATE',  # 기존 데이터 지우고 새로 넣기
)

job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
job.result()  # 완료될 때까지 대기

print(f'적재 완료: {job.output_rows}행')

E:\Data_Project\public-data-analysis\.venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


적재 완료: 37987행


In [17]:
df.columns

Index(['sd_code', 'year', 'month', 'sd_name', 'sgg_name', 'house_cnt',
       'avg_usage_kwh', 'avg_bill_won', 'quarter', 'season', 'season_tariff',
       'total_usage_kwh', 'total_bill_won', 'unit_price', 'usage_mom_pct',
       'usage_yoy_pct'],
      dtype='str')